# Travel Assistant: a chat that remembers, looks things up, and checks itself

**What we build:** a small assistant that checks weather, suggests places and searches flights, and understands follow-ups like *"What can I do **there**?"* At the end, **you chat with it live**.

**Two problems, two fixes**

| Problem | Fix |
|---|---|
| A model cannot see our data | give it **tools** (normal Python functions it may ask us to run) |
| A model forgets every message | send the **history** (earlier messages) back each time |

**Four steps** (we run something after each one):

1. Load the data
2. Make the tools
3. Build the chain
4. Run the loop, then ask six questions and chat live

**Four small touches that make it more than a demo**

* **Exact logic lives in code:** a route planner finds one-stop trips, so the model never has to guess.
* **Guards:** a step limit and safe tool errors, so one bad call cannot crash or loop forever.
* **A trace line** after every answer: which tools ran and how many tokens were used.
* **A scorecard** that checks the six answers automatically.

All data is fake and lives inside this notebook. No travel API is needed.

## Step 0: Setup

Install LangChain and give the notebook access to the **TrueFoundry LLM gateway**.

Create a `.env` file next to this notebook (never commit it) with:

```
LLM_GATEWAY_BASE_URL=https://truefoundry.innovaccer.com/api/llm/api/inference/openai/
LLM_GATEWAY_CLAUDE=<your TrueFoundry gateway token>
LLM_MODEL=internal-bedrock/sonnet-46
```

The next cell loads these with `python-dotenv`.

In [ ]:
!pip -q install langchain langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()                                   # reads .env in this folder

REQUIRED = ["LLM_GATEWAY_BASE_URL", "LLM_GATEWAY_CLAUDE", "LLM_MODEL"]
missing = [key for key in REQUIRED if not os.environ.get(key)]
if missing:
    raise RuntimeError(f"Missing environment variables: {missing}. Add them to a .env file next to this notebook.")

## Step 1: Load the data

Our "database" is three plain Python variables. Weather and attractions are looked up **by city**; flights are a **list** we filter.

In [ ]:
FAKE_WEATHER_DB = {
    "Delhi": {"condition": "Hot and sunny", "temp_c": 36, "rain_chance": 20},
    "Jaipur": {"condition": "Sunny", "temp_c": 34, "rain_chance": 10},
    "Mumbai": {"condition": "Heavy clouds", "temp_c": 29, "rain_chance": 75},
    "Bengaluru": {"condition": "Pleasant with light rain", "temp_c": 25, "rain_chance": 55},
    "Goa": {"condition": "Cloudy", "temp_c": 28, "rain_chance": 65},
}

FAKE_ATTRACTIONS_DB = {
    "Delhi": [
        {"name": "India Gate", "type": "Landmark", "time_needed": "1 hour"},
        {"name": "Humayun's Tomb", "type": "Heritage", "time_needed": "2 hours"},
    ],
    "Jaipur": [
        {"name": "Amer Fort", "type": "Heritage", "time_needed": "3 hours"},
        {"name": "Hawa Mahal", "type": "Landmark", "time_needed": "1 hour"},
    ],
    "Mumbai": [
        {"name": "Gateway of India", "type": "Landmark", "time_needed": "1 hour"},
        {"name": "Marine Drive", "type": "Leisure", "time_needed": "2 hours"},
    ],
    "Bengaluru": [
        {"name": "Cubbon Park", "type": "Nature", "time_needed": "2 hours"},
        {"name": "Bangalore Palace", "type": "Heritage", "time_needed": "2 hours"},
    ],
    "Goa": [
        {"name": "Basilica of Bom Jesus", "type": "Heritage", "time_needed": "2 hours"},
        {"name": "Baga Beach", "type": "Leisure", "time_needed": "3 hours"},
    ],
}

FAKE_FLIGHTS_DB = [
    {"flight": "AI-101", "from": "Delhi", "to": "Jaipur", "departure": "08:10", "price_inr": 4200, "seats": 6},
    {"flight": "6E-221", "from": "Delhi", "to": "Jaipur", "departure": "18:45", "price_inr": 3900, "seats": 3},
    {"flight": "AI-305", "from": "Delhi", "to": "Mumbai", "departure": "09:30", "price_inr": 6100, "seats": 8},
    {"flight": "UK-822", "from": "Mumbai", "to": "Goa", "departure": "13:20", "price_inr": 3500, "seats": 4},
    {"flight": "6E-517", "from": "Bengaluru", "to": "Goa", "departure": "07:40", "price_inr": 3200, "seats": 5},
    {"flight": "AI-640", "from": "Jaipur", "to": "Bengaluru", "departure": "16:15", "price_inr": 7200, "seats": 2},
]

**Check it by hand first**, so we trust the data before a model touches it:

In [ ]:
FAKE_WEATHER_DB["Jaipur"]

## Step 2: Make the tools

A **tool** (the slides call it a *helper*) is a normal Python function that the model is allowed to *ask us to run*. The model never touches our data. It asks, our code looks it up, and the answer goes back.

The model chooses a tool by reading its **name** and its **one-line description** (the text in triple quotes). Clear descriptions mean the right tool gets picked.

The fourth tool, `plan_route`, is our own idea. When there is no direct flight, **joining two flights is exact work**, so we do it in code and let the model do what it is good at: language.

In [ ]:
import json
from langchain_core.tools import tool

ALIASES = {"bangalore": "Bengaluru"}          # one friendly nickname

def clean_city(name: str) -> str:
    """Make 'jaipur ' or 'Bangalore' match our data."""
    name = name.strip()
    return ALIASES.get(name.lower(), name.title())


@tool
def get_weather(city: str) -> str:
    """Get the current weather for one city: condition, temperature in C and chance of rain."""
    city = clean_city(city)
    if city not in FAKE_WEATHER_DB:
        return f"No weather data for {city}. Cities we cover: {list(FAKE_WEATHER_DB)}"
    return json.dumps({"city": city, **FAKE_WEATHER_DB[city]})


@tool
def get_attractions(city: str) -> str:
    """List the attractions in one city, with type and time needed."""
    city = clean_city(city)
    if city not in FAKE_ATTRACTIONS_DB:
        return f"No attraction data for {city}. Cities we cover: {list(FAKE_ATTRACTIONS_DB)}"
    return json.dumps({"city": city, "attractions": FAKE_ATTRACTIONS_DB[city]})


@tool
def search_flights(origin: str, destination: str) -> str:
    """Find DIRECT flights from one city to another: flight number, departure time, price in INR, seats left."""
    origin, destination = clean_city(origin), clean_city(destination)
    found = [f for f in FAKE_FLIGHTS_DB if f["from"] == origin and f["to"] == destination]
    if not found:
        return f"No direct flight from {origin} to {destination} in our inventory."
    return json.dumps(found)


@tool
def plan_route(origin: str, destination: str) -> str:
    """Find routes with ONE stop when there is no direct flight. Returns both flights, the total price in INR and whether the connection works."""
    origin, destination = clean_city(origin), clean_city(destination)
    routes = []
    for first in FAKE_FLIGHTS_DB:
        for second in FAKE_FLIGHTS_DB:
            if first["from"] == origin and first["to"] == second["from"] and second["to"] == destination:
                if second["departure"] > first["departure"]:
                    note = "second flight leaves later the same day, but we have no arrival times, so the gap is unconfirmed"
                else:
                    note = "second flight leaves earlier in the day, so this needs an overnight stay"
                routes.append({"first": first, "second": second,
                               "total_price_inr": first["price_inr"] + second["price_inr"], "connection": note})
    if not routes:
        return f"No one-stop route from {origin} to {destination} in our inventory."
    return json.dumps(routes)


TOOLS = [get_weather, get_attractions, search_flights, plan_route]

**Test each tool alone**, with no model involved:

In [ ]:
print(get_weather.invoke({"city": "Jaipur"}))
print(search_flights.invoke({"origin": "Delhi", "destination": "Jaipur"}))
print(plan_route.invoke({"origin": "Jaipur", "destination": "Goa"}))
print(get_weather.invoke({"city": "Chennai"}))

## Step 3: Build the chain

A **chain** connects pieces in a fixed order, so the output of one feeds the next. Ours has two pieces:

```
prompt  ->  model (which knows our tools)
```

* The **prompt** holds our instructions plus a slot called `messages`. That slot is where the **history** goes.
* The **model** reads everything and either answers or asks for a tool.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

MODEL = os.environ["LLM_MODEL"]                 # TrueFoundry virtual model, e.g. internal-bedrock/sonnet-46

model = ChatOpenAI(
    base_url=os.environ["LLM_GATEWAY_BASE_URL"],
    api_key=os.environ["LLM_GATEWAY_CLAUDE"],
    model=MODEL,
)
model_with_tools = model.bind_tools(TOOLS)      # tell the model which tools exist

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a friendly travel assistant for a small demo company. "
     "Use your tools for weather, attractions and flights, and answer only from what they return. "
     "If there is no direct flight, use plan_route. "
     "If the data has no answer, say so plainly. Flights list departure times only, so never invent arrival times. "
     "When the user says 'there' or 'that one', look at the earlier messages. Keep answers short."),
    MessagesPlaceholder("messages"),            # the history goes here
])

chain = prompt | model_with_tools               # <- the chain

## Step 4: Run the loop

One user question can need more than one trip to the model, so we write a small loop:

1. Send the messages to the chain.
2. If the model **asks for a tool**, run it, add the result to the messages, and go back to step 1.
3. If the model **just answers**, print the answer and stop.

`history` is simply a Python list. After each question we save the whole conversation into it. That list *is* the memory.

Two guards keep it safe: the loop stops after `MAX_STEPS` trips, and a failing tool returns an error message instead of crashing.
After every answer, a **trace line** shows the tools used and the tokens spent. Watch the token count grow as the history grows.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

TOOL_BY_NAME = {t.name: t for t in TOOLS}
MAX_STEPS = 5                                   # guard: never loop forever
history = []                                    # the memory: a plain list of messages
LOG = {}                                        # one record per tagged question, for the scorecard


def text_of(message):
    """The reply text (some models return a list of blocks)."""
    if isinstance(message.content, str):
        return message.content
    return "".join(b.get("text", "") for b in message.content if isinstance(b, dict))


def ask(question, use_history=True, tag=None, echo=True):
    messages = (history if use_history else []) + [HumanMessage(question)]
    tools_used, tokens_in, tokens_out = [], 0, 0
    if echo:
        print(f"You: {question}")

    for _ in range(MAX_STEPS):
        reply = chain.invoke({"messages": messages})        # 1. ask the model
        messages.append(reply)
        usage = reply.usage_metadata or {}
        tokens_in += usage.get("input_tokens", 0)
        tokens_out += usage.get("output_tokens", 0)
        if not reply.tool_calls:                            # 3. plain answer: done
            break
        for call in reply.tool_calls:                       # 2. run each requested tool
            print(f"   [tool] {call['name']}({call['args']})")
            tools_used.append(call["name"])
            try:
                result = TOOL_BY_NAME[call["name"]].invoke(call["args"])
            except Exception as error:                      # guard: a bad call must not crash us
                result = f"Tool error: {error}"
            messages.append(ToolMessage(content=result, tool_call_id=call["id"]))

    answer = text_of(reply) or "(no answer: step limit reached)"
    if use_history:
        history[:] = messages                               # remember everything
    if tag:
        LOG[tag] = {"tools": tools_used, "answer": answer}

    print(f"Assistant: {answer}")
    print(f"   [trace] tools: {tools_used or 'none'} | tokens in/out: {tokens_in}/{tokens_out} | memory: {len(history)} messages\n")


def show_history():
    """Print what the model will get on the next turn."""
    for m in history:
        kind = type(m).__name__
        note = f"asked {[c['name'] for c in m.tool_calls]}" if getattr(m, "tool_calls", None) else text_of(m)[:70]
        print(f"{kind:13} {note}")

## The conversation: six questions

Run these **in order**. Each one builds on the last.

**1. A first question.** The model should pick the *weather* tool.

In [ ]:
ask("Is Jaipur a good option this weekend? What's the weather like?", tag="Q1")

**2. Memory: first with no history, then with it.**
"there" means nothing on its own. Run the first cell and watch it fail, then run the second.

In [ ]:
ask("What can I do there?", use_history=False)      # no memory

In [ ]:
ask("What can I do there?", tag="Q2")               # with memory

**3. A different tool, plus comparing.** The model reads the flight results and picks the cheaper one.

In [ ]:
ask("How do I get there from Delhi? Which is cheaper?", tag="Q3")

**4. Answer from memory.** No new tool call: the flight details are already in the history.

In [ ]:
ask("Which one leaves in the evening?", tag="Q4")

**5. Joining two flights.** There is no direct Jaipur to Goa flight. Watch it call our route planner and **avoid inventing arrival times**.

In [ ]:
ask("Can I continue to Goa from there?", tag="Q5")

**6. Saying "I don't know".** Chennai is not in our data, so a good answer admits it.

In [ ]:
ask("What about Chennai?", tag="Q6")

## Did it behave? A tiny scorecard

A demo that works once proves little. These checks look at *what the assistant did*, not just what it said.

A model can vary between runs, so a ❌ is information, not an error. Re-run the question and see.

In [ ]:
import re

CHECKS = {
    "Q1 used the weather tool":        lambda r: "get_weather" in r["tools"],
    "Q2 found attractions via memory": lambda r: "get_attractions" in r["tools"],
    "Q3 searched flights":             lambda r: "search_flights" in r["tools"],
    "Q4 answered from memory":         lambda r: r["tools"] == [],
    "Q5 planned a one-stop route":     lambda r: "plan_route" in r["tools"],
    "Q6 invented no weather numbers":  lambda r: not re.search(r"\d+\s*(°|degrees|C\b)", r["answer"]),
}

for name, check in CHECKS.items():
    record = LOG.get(name[:2])
    status = "not run" if record is None else ("✅" if check(record) else "❌")
    print(f"{status}  {name}")

## Your turn: chat with it

Type your own questions. Try follow-ups, cities we do not cover, or routes that need a stop.

Commands: `/history` shows the memory, `/reset` wipes it, `/quit` stops.

In [ ]:
def chat():
    print("Ask about weather, places and flights.  /history  /reset  /quit\n")
    while True:
        question = input("You: ").strip()
        if not question:
            continue
        if question == "/quit":
            break
        if question == "/reset":
            history.clear()
            print("Memory cleared.\n")
        elif question == "/history":
            show_history()
            print()
        else:
            ask(question, echo=False)

chat()

## Recap

1. A model forgets, so we **send the history back**.
2. A model cannot see our data, so we give it **tools**, and keep exact logic (like routes) in code.
3. A **chain** (prompt, then model) plus a small **loop** connects question, tool and answer.
4. Guards, a trace and a scorecard make it something you can **trust and debug**.

**Where this goes next:** history grows, so real systems trim or summarise it; the fake data becomes a real API or a document search; the scorecard becomes a regular test.